# 04 · Validate — BindCraft vs RFdiffusion + isoform specificity + effector competition

**Standard slot:** *validate (in silico).* **For Project 08 this is the core analysis:** the
head-to-head between the two paradigms (hit rate, interface energy, novelty), **the
isoform-specificity panel (KRAS vs HRAS/NRAS, the selectivity gap)** — the scientific heart of this
project — and **effector-competition reasoning (block RAF)** `[extension]`, with publication-style
figures (D3 part 2).

Needs `results/bindcraft_designs.csv` + `results/rfdiffusion_designs.csv` + `results/all_ranked.csv`
(from notebooks 02–03).

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Head-to-head hit rate + interface energy

Compare the two paradigms on (a) all-layers **hit rate** and (b) the **interface-energy** (`rosetta_dG`)
distribution of survivors. A fair comparison filters both identically (notebook 03) and reports the
*distribution*, not the single best. Mock numbers are SYNTHETIC.

In [ ]:
import pandas as pd, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ranked = pd.read_csv("results/all_ranked.csv")
print("paradigms:", ranked["paradigm"].value_counts().to_dict())

summary = []
for p, g in ranked.groupby("paradigm"):
    n = len(g); passed = int((g["layers_passed"] >= 3).sum())
    summary.append(dict(paradigm=p, n=n, all_layers_survivors=passed,
                        hit_rate_pct=round(100*passed/max(n,1), 1),
                        median_pae=round(float(g["pae_interaction"].median()), 2),
                        median_dG=round(float(g["rosetta_dG"].median()), 2)))
summary = pd.DataFrame(summary)
print("\nhead-to-head summary (SYNTHETIC if mock):")
print(summary.to_string(index=False))

In [ ]:
# Interface-energy distribution per paradigm (survivors).
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for p, g in ranked.groupby("paradigm"):
    surv = g[g["layers_passed"] >= 3]
    ax[0].hist(g["pae_interaction"].dropna(), bins=15, alpha=0.5, label=p)
    ax[1].hist(surv["rosetta_dG"].dropna(), bins=15, alpha=0.5, label=p)
ax[0].set_xlabel("pae_interaction (Å, lower better)"); ax[0].set_ylabel("designs"); ax[0].set_title("AF2-Multimer pae_interaction"); ax[0].legend()
ax[1].set_xlabel("rosetta_dG (REU, more negative better)"); ax[1].set_title("Interface energy (survivors)"); ax[1].legend()
fig.suptitle("BindCraft vs RFdiffusion vs KRAS (EXAMPLE_DATA if mock)")
plt.tight_layout(); plt.savefig("results/p08_headtohead.png", dpi=150); plt.show()
print("saved results/p08_headtohead.png")

## 2 · Isoform-specificity panel — KRAS vs HRAS/NRAS (the centerpiece)

The real question for KRAS is **selectivity**: a binder that hits KRAS *and* HRAS/NRAS is far less
useful (and more toxic) than a selective one. For each survivor we re-score the binder against each RAS
isoform with the **same** AF2-Multimer scorer (`specificity_panel`, mock here) and compute the
**selectivity gap** = KRAS pae − best off-target pae (positive ⇒ KRAS scores better ⇒ some selectivity).
Because the isoforms are nearly identical across the switches, **expect this to be hard** — report the
gap distribution honestly, including binders that turn out pan-RAS. On Colab, replace `tool="mock"`
with `tool="af2"` and verified HRAS/NRAS structures.

In [ ]:
import binder_tools as bt

# Rebuild the survivor sequences from the pools (we need the sequence to re-score vs each isoform).
bc = pd.read_csv("results/bindcraft_designs.csv")
rf = pd.read_csv("results/rfdiffusion_designs.csv")
pools = pd.concat([bc, rf], ignore_index=True).set_index("design_id")

surv = ranked[ranked["layers_passed"] >= 3].copy()
rows = []
for _, r in surv.iterrows():
    did = r["design_id"]
    if did not in pools.index:
        continue
    seq = str(pools.loc[did, "sequence"])
    # Reconstruct a lightweight BinderDesign so the helper has allele/state/hotspots context.
    d = bt.BinderDesign(design_id=did, sequence=seq, paradigm=r["paradigm"], target="KRAS",
                        allele=str(pools.loc[did].get("allele", "WT")),
                        nucleotide_state=str(pools.loc[did].get("nucleotide_state", "GDP")),
                        hotspots=bt.parse_hotspots(str(pools.loc[did].get("contact_residues", "") or "")))
    panel = bt.specificity_panel(d, tool="mock")   # -> tool="af2" on Colab with verified HRAS/NRAS
    rows.append(dict(design_id=did, paradigm=r["paradigm"],
                     kras_pae=panel["kras_pae"], best_offtarget_pae=panel["best_offtarget_pae"],
                     selectivity_gap=panel["selectivity_gap"], selective=panel["selective"]))
spec = pd.DataFrame(rows)
spec.to_csv("results/isoform_specificity.csv", index=False)
print("wrote results/isoform_specificity.csv", spec.shape, "(SYNTHETIC if mock)")
if len(spec):
    for p, g in spec.groupby("paradigm"):
        print(f"  {p:12s}: median selectivity_gap = {g['selectivity_gap'].median():.2f}  "
              f"selective fraction = {g['selective'].mean():.2f}  (n={len(g)})")
    print("\nA gap ~ 0 ⇒ pan-RAS (binds HRAS/NRAS too) — a weaker result. Report the full distribution.")

In [ ]:
# Selectivity-gap distribution per paradigm.
if len(spec):
    fig, ax = plt.subplots(figsize=(6, 3.4))
    for p, g in spec.groupby("paradigm"):
        ax.hist(g["selectivity_gap"].dropna(), bins=15, alpha=0.5, label=p)
    ax.axvline(0, color="k", ls="--", lw=1, label="pan-RAS (gap=0)")
    ax.set_xlabel("selectivity gap = KRAS pae − best off-target pae (Å; >0 = selective)")
    ax.set_ylabel("survivors"); ax.set_title("Isoform selectivity (EXAMPLE_DATA if mock)"); ax.legend()
    plt.tight_layout(); plt.savefig("results/p08_selectivity.png", dpi=150); plt.show()
    print("saved results/p08_selectivity.png")
else:
    print("No survivors to plot — loosen the dry-run sizes or check notebook 03.")

## 3 · Novelty `[extension]`

Novelty = TM-score of each binder to its nearest natural fold (Foldseek/TM-align; `< 0.5` ≈ novel). On
Colab, compute it per design and compare the two paradigms' novelty distributions. Here we scaffold the
analysis (mock has no real structures), so we just show where it plugs in.

In [ ]:
# Scaffold: on Colab, run Foldseek/TM-align on each predicted binder backbone -> tm_to_pdb,
# then compare distributions across paradigms (novel == tm_to_pdb < 0.5).
if "tm_to_pdb" in ranked.columns and ranked["tm_to_pdb"].notna().any():
    for p, g in ranked.groupby("paradigm"):
        novel = (g["tm_to_pdb"] < 0.5).mean()
        print(f"{p:12s}: novel fraction (TM<0.5) = {novel:.2f}")
else:
    print("Novelty scaffold — populate tm_to_pdb with Foldseek/TM-align on Colab, then compare paradigms.")

## 4 · Effector-competition vs RAF `[extension]`

A switch-region binder could **block RAF/effector engagement** if it covers enough of the switch
footprint. `hotspot_overlap` (notebook 02) is our geometry proxy: the fraction of switch I/II hotspots
the binder contacts. Higher ⇒ more likely to occlude the effector interface. Compare the survivors'
coverage across paradigms — a strong interface that *misses* the switch regions won't block RAF. (On
Colab, model the binder + RAF-RBD competition directly for a stronger test.)

In [ ]:
bc = pd.read_csv("results/bindcraft_designs.csv")
rf = pd.read_csv("results/rfdiffusion_designs.csv")
pools2 = pd.concat([bc, rf], ignore_index=True)

ov = pools2.set_index("design_id")["hotspot_overlap"]
ranked["hotspot_overlap"] = ranked["design_id"].map(ov)
surv2 = ranked[ranked["layers_passed"] >= 3]

print("switch-footprint coverage of all-layers survivors (SYNTHETIC if mock):")
for p, g in surv2.groupby("paradigm"):
    print(f"  {p:12s}: median switch-footprint coverage = {g['hotspot_overlap'].median():.2f}  (n={len(g)})")

# "Likely effector blockers" = survivors that also cover enough of the switch footprint.
BLOCK_OVERLAP = 0.5
blockers = surv2[surv2["hotspot_overlap"] >= BLOCK_OVERLAP]
print(f"\nlikely effector blockers (survivor AND switch coverage>={BLOCK_OVERLAP}): {len(blockers)}")
print(blockers.groupby("paradigm").size().to_dict())

## 5 · Select the top 10–20 per paradigm (prefer selective blockers)

The D★ deliverable wants the **top 10–20 each**. Rank survivors by the composite score and, as
tie-breakers, prefer a higher **selectivity gap** (KRAS-selective) and higher switch-footprint coverage
(effector blocker). Save the shortlist for the validation plan (notebook 05).

In [ ]:
# Join the selectivity gap onto the ranked survivors.
if len(spec):
    gap = spec.set_index("design_id")["selectivity_gap"]
    ranked["selectivity_gap"] = ranked["design_id"].map(gap)
else:
    ranked["selectivity_gap"] = np.nan

top_per = []
for p, g in ranked.groupby("paradigm"):
    g2 = g[g["layers_passed"] >= 3].sort_values(
        ["score", "selectivity_gap", "hotspot_overlap"], ascending=False).head(20)
    top_per.append(g2)
top = pd.concat(top_per, ignore_index=True)
top.to_csv("results/top_candidates.csv", index=False)
print("wrote results/top_candidates.csv:", top.shape, "(top<=20 per paradigm)")
print(top.groupby("paradigm").size().to_dict())
top.head(8)[["design_id", "paradigm", "score", "pae_interaction", "rosetta_dG",
             "selectivity_gap", "hotspot_overlap"]]

## D3 (part 2) checklist
- [ ] Head-to-head: hit rate + interface-energy distribution per paradigm (figure `results/p08_headtohead.png`).
- [ ] **Isoform-specificity panel: selectivity gap per survivor + per paradigm** (`results/isoform_specificity.csv`, `results/p08_selectivity.png`).
- [ ] Novelty compared across paradigms (TM-score to PDB) — or the scaffold wired up on Colab.
- [ ] Effector-competition vs RAF: switch-footprint coverage of survivors; "likely blocker" count.
- [ ] `results/top_candidates.csv`: top 10–20 each, preferring selective blockers, ready for the validation plan.
- [ ] Honest discussion: the two paradigms' failure modes **and** why isoform selectivity is hard (not just a winner).

**Next:** `05_validation_plan.ipynb` — the SPR/BLI + isoform-specificity + nucleotide-state plan.